In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
from tqdm import tqdm

In [2]:
def generate_anchors_positions(positions_segments, attach):
    positions_anchors = np.zeros((len(attach), 3))

    for idx, a in enumerate(attach):
        positions_anchors[idx] = np.random.normal(positions_segments[a], np.sqrt(Re**2/(3*(N-1)*beta)))
        
        # print(idx, a, positions_segments[a])
    return positions_anchors

def bonded_energy(positions_segments):
    u = 0
    for segment in range(1, len(positions_segments)):
        u += np.linalg.norm(positions_segments[segment] - positions_segments[segment-1])**2

    return u

def slipSpring_energy(positions_segments, positions_anchors, anchor):
    w = 0
    
    for idx, a in enumerate(anchor):
        if a == -1:
            continue
        else:
            w += np.linalg.norm(positions_segments[idx] - positions_anchors[a])**2
        # print(a, positions_segments[idx], positions_anchors[a])
    return w

def hamiltonian(positions_segments, positions_anchors, anchor):
    h = bonded_energy(positions_segments) + beta*slipSpring_energy(positions_segments, positions_anchors, anchor)
    h *= -3*(N-1)/(2*Re**2)
    return h

In [3]:
def plotter(positions_segments, anchor, positions_anchors, step):
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')

    # Chain backbone
    ax.plot(positions_segments[:, 0], positions_segments[:, 1], positions_segments[:, 2], "-", color="k", alpha=0.5)

    # Beads
    for segment in range(len(positions_segments)):
        if anchor[segment] == -1:
            ax.scatter(*positions_segments[segment], color="k", facecolors="none", alpha=0.8)
        else:
            ax.scatter(*positions_segments[segment], color="k", alpha=0.8)

    # Chain ends
    ax.scatter(*positions_segments[0], color="b", marker="s", s=60, facecolors="none", label="bead N=1")
    ax.scatter(*positions_segments[-1], color="g", marker="s", s=60, facecolors="none", label=f"bead N={len(positions_segments)}")

    # Anchor connections
    for idx, a in enumerate(anchor):
        if a != -1:
            ax.plot(
                [positions_anchors[a, 0], positions_segments[idx, 0]],
                [positions_anchors[a, 1], positions_segments[idx, 1]],
                [positions_anchors[a, 2], positions_segments[idx, 2]],
                color='red', linestyle='-', linewidth=1, alpha=0.4
            )
            ax.scatter(
                *positions_anchors[a],
                color='red', marker="s", alpha=0.4, s=10
            )

    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    ax.legend(loc="upper right")
    ax.grid(False)

    ax.set_title(f"step {step}")
    fig.tight_layout()

    output_filepath = f"data/imgs/mcmc/evolution/"
    output_filename = f"step_{step}.png"
    
    if not os.path.exists(output_filepath):
        os.makedirs(output_filepath)
    file = os.path.join(output_filepath, output_filename)
    
    plt.savefig(file, dpi=300)
    plt.close()

In [27]:
# System Parameters
kB = 1
T = 1


N = 16 # Number of segments
Z = N//4 # Number of slip links

Re = 1 # Polymer end-to-end distance
beta = 0.5 # Anchor-segment stiffness (?)

dims = 3 # Space dimensions


a = 2*Re/np.sqrt(N-1) # bead "exploratory" linear dimension -> any proposal confined within such a cube

N_steps = 100 # MC steps

In [28]:
# Initialize the system

# Initialize polymer
positions_segments = np.zeros((N, dims))

# whereby, assumed that the first segment is initialized at (0, 0, 0)
for segment in range(1, N):
    # print(segment)
    positions_segments[segment] = positions_segments[segment-1] + np.random.normal(0, 1, dims)

# Beads with slip-springs
# Get the index of the beads with the slip spring
attach = np.random.choice(N, Z, replace=False)

# "idx-ordered" list of slip springs (similar to a hash table)
anchor = -1*np.ones(N, dtype=np.int64)
for idx, link in enumerate(attach):
    anchor[link] = idx

positions_anchors = generate_anchors_positions(positions_segments, attach)

# Visualize initial config.
plotter(positions_segments, anchor, positions_anchors, step=0)

In [29]:
# sequential: exhaust segment moves first and then exhause slip-links moves
for step in range(N_steps):
    plotter(positions_segments, anchor, positions_anchors, step)
    
    H_current=hamiltonian(positions_segments, positions_anchors, anchor)
    
    # MC on segment
    for _ in range(N):
        segment = np.random.randint(0, N) # segment chosen at random
        
        old_pos = positions_segments[segment].copy()
        
        positions_segments[segment] += np.random.uniform(-a, a, 3)
        H_proposal = hamiltonian(positions_segments, positions_anchors, anchor)
        H_delta = H_proposal - H_current

        if H_delta <= 0 or np.random.rand() < np.exp(-H_delta / (kB * T)):
            H_current = H_proposal
        else:
            positions_segments[segment] = old_pos


    # MC on slip-links
    for _ in range(Z):
        j = np.random.randint(0, Z) # slip-link chosen at random
        
        current_attachment = attach[j].copy() # choosing the random attachment point
        proposal_attachment = current_attachment + np.random.choice([-1, 1])

        anchor[current_attachment] = -1 # remove attachment at the chosen j idx
        anchor[proposal_attachment] = j # add attachment at the proposed j idx
        attach[j] = proposal_attachment

        H_proposal = hamiltonian(positions_segments, positions_anchors, anchor)
        H_delta = H_proposal - H_current

        if H_delta <= 0 or np.random.rand() < np.exp(-H_delta / (kB * T)):
            H_current = H_proposal
        else:
            anchor[current_attachment] = j
            anchor[proposal_attachment] = -1
            attach[j] = current_attachment
    #     print(current_attachment, proposal_attachment)
    # print()

11 12
12 11
12 13
10 9

9 8
12 11
4 5
9 10

9 8
5 4
13 12
5 4

5 4
13 12
13 14
11 12

9 8
9 8
9 10
14 13

9 10
5 6
5 6
14 15

9 10
11 12
9 10
9 8

11 10
5 6
9 8
9 10

9 10
14 13
9 8
5 4

14 13
10 11
11 12
11 10

5 6
14 13
14 13
9 8

14 13
5 4
5 4
5 6

14 15
14 13
8 7
8 7

14 15
8 7
8 9
14 15

10 11
10 11
14 13
10 11

8 7
10 11
8 9
5 4

5 4
14 15
14 15
14 13

14 15
10 9
9 10
9 10

5 4
14 15
14 13
5 6

14 15
5 6
8 9
9 8

9 8
8 7
5 4
14 13

5 4
14 15
5 4
8 7

5 6
5 4
8 9
5 6

14 15
8 9
8 9
5 4

8 9
14 13
8 7
14 13

8 7
14 15
8 9
8 9

5 4
8 9
14 13
14 15

14 15
8 9
8 7
8 9

14 13
14 15
14 15
5 6

14 15
8 7
5 4
14 13

8 7
5 6
14 13
8 7

8 7
14 15
8 9
14 15

5 6
8 9
8 9
8 7

5 4
5 4
14 13
8 7

8 7
8 9
8 7
5 6

5 6
8 7
8 7
5 6

5 6
8 7
8 9
8 7

5 4
5 6
14 15
5 4

8 7
8 7
8 9
8 9

5 4
14 15
8 9
5 4

8 9
14 13
8 9
8 9

8 9
14 13
5 6
8 7

8 7
8 7
14 15
14 13

5 4
8 9
8 9
8 9

14 15
14 13
8 7
8 7

14 15
14 13
14 13
14 15

14 15
8 7
5 4
8 7

8 9
14 15
8 7
8 9

5 6
5 4
8 7
8 7

8 9
8 9
14 15
5 6

5